# Case Study

This has been simplified from course materials at https://github.com/uc-python/advanced-python-datasci.

For the purposes of this project, we are only interested in obtaining a model. However, if you would like to see a more thorough explanation of *how* the model was decided upon, please check out the link above.


In [7]:
import pandas as pd
import numpy as np

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def get_features_and_target(csv_file, target_col):
    '''Split a CSV into a DF of numeric features and a target column.'''
    
    adult_census = pd.read_csv(csv_file)
    
    raw_features = adult_census.drop(columns=target_col)
    numeric_features = raw_features.select_dtypes(np.number)
    feature_cols = numeric_features.columns.values

    features = adult_census[feature_cols]
    target = adult_census[target_col]
    
    return (features, target)

def make_preprocessor(features, categorical_preprocessor=None, numeric_preprocessor=None):
    '''Create a column transformer that applies sensible preprocessing procedures.'''
    
    if categorical_preprocessor is None:
        categorical_preprocessor = OneHotEncoder(handle_unknown='ignore')
    if numeric_preprocessor is None:
        numeric_preprocessor = StandardScaler()
        
    numeric_columns = features.select_dtypes(exclude=object).columns
    categorical_columns = features.select_dtypes(include=object).columns
    preprocessor = ColumnTransformer([
        ('one-hot-encoder', categorical_preprocessor, categorical_columns),
        ('standard_scaler', numeric_preprocessor, numeric_columns)
    ])
    return preprocessor

In [9]:
num_features, target = get_features_and_target('./data/ames.csv', 'Sale_Price')

Split the data into training and test sets. Use 75% of the data for training and 25% for testing.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    num_features, 
    target, 
    random_state=123, 
    test_size=0.25
)

In [11]:
X_test.iloc[-10:].to_json('./data/test.json', orient='records')

2. Fit a default `KNeighborsRegressor` model on the training data and score on the test data. Note that scoring on regression models provides the $R^2$.

In [12]:
from sklearn.neighbors import KNeighborsRegressor

# 1. define the algorithm
knn_model = KNeighborsRegressor()

# 2. fit the model
knn_model.fit(X_train, y_train)

# 3. score our model on test data
knn_model.score(X_test, y_test)

0.6949157417705423

3. Fit a default `sklearn.linear_model.LinearRegression` model on the training data and score on the test data.

In [13]:
from sklearn.linear_model import LinearRegression

# 1. define the algorithm
lm_model = LinearRegression()

# 2. fit the model
lm_model.fit(X_train, y_train)

# 3. score our model on test data
lm_model.score(X_test, y_test)

0.8104251490010677

4. Fit a default `sklearn.ensemble.RandomForestRegressor` model on the training data and score on the test data.

In [14]:
from sklearn.ensemble import RandomForestRegressor

# 1. define the algorithm
rf_model = RandomForestRegressor()

# 2. fit the model
rf_model.fit(X_train, y_train)

# 3. score our model on test data
rf_model.score(X_test, y_test)

0.8790991811245455

### Feature engineering

1. Fill in the blanks to standardize the numeric features and then apply a linear regression model. Does standardizing the numeric features improve the linear regression's $R^2$?

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lm_model_scaled = make_pipeline(StandardScaler(), LinearRegression())
lm_model_scaled.fit(X_train, y_train)
lm_model_scaled.score(X_test, y_test)

0.8104251490010679

2. Using the code chunks below, which computes the following:

- identifies numeric, categorical, and ordinal columns in our full feature set,
- replaces unique values in our ordinal columns (i.e. "No_basement", "No_garage"), and
- creates our encoders for the numeric, categorical, and ordinal columns.

In [16]:
# get columns of interest
ames =  pd.read_csv('./data/ames.csv')
features = ames.drop(columns='Sale_Price')
cat_features = features.select_dtypes(include=object)
numerical_columns = num_features.columns
ordinal_columns = cat_features.filter(regex='Qual').columns
categorical_columns = cat_features.drop(columns=ordinal_columns).columns

# replace unique values in our ordinal columns (i.e. "No_basement", "No_garage") with 'NA'
for col in ordinal_columns:
    features[col] = features[col].replace(to_replace='No_.*', value='NA', regex=True)
    
# split full feature set (numeric, categorical, & ordinal features) into train & test sets
X_train, X_test, y_train, y_test = train_test_split(features, target, random_state=123)

In [17]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# create our numeric, categorical, and ordinal preprocessor encoders
numerical_preprocessor = StandardScaler()
categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")

ordinal_categories = [
    "NA", "Very_Poor", "Poor", "Fair", "Below_Average", "Average", "Typical",
    "Above_Average", "Good", "Very_Good", "Excellent", "Very_Excellent"
]
list_of_ord_cats = [ordinal_categories for col in ordinal_columns]
ordinal_preprocessor = OrdinalEncoder(categories=list_of_ord_cats)

2. Continued...

Now fill in the blanks to create our `ColumnTransformer` that:

- standardizes numerical columns
- one-hot encodes categorical columns
- ordinal encodes ordinal columns

In [18]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('standard_scaler', numerical_preprocessor, numerical_columns),
    ('one_hot_encoder', categorical_preprocessor, categorical_columns),
    ('ordinal_encoder', ordinal_preprocessor, ordinal_columns),
])

3. Now create a pipeline that includes the preprocessing step and applies a linear regression model. Does this improve the linear regression's $R^2$?

In [19]:
lm_full = make_pipeline(preprocessor, LinearRegression())
_ = lm_full.fit(X_train, y_train)
lm_full.score(X_test, y_test)

0.884969702076682

4. apply these preprocessing steps with a default random forest model and see if performance improves.

In [20]:
rf_full = make_pipeline(preprocessor, RandomForestRegressor())
_ = rf_full.fit(X_train, y_train)
rf_full.score(X_test, y_test)

0.9026842589377609

In [21]:
from joblib import dump

dump(rf_full,'model.joblib')

['model.joblib']

In [23]:
X_train[-3:].to_json('./data/X_test.json', orient='records')